In [ ]:
import os, sys, torch, gc, time
from tqdm.notebook import tqdm
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

print("Imports restored! ✓")

Imports restored! ✓


In [ ]:
!unzip checkpoints.zip -d /teamspace/studios/this_studio/checkpoints

In [3]:
!pip install "setuptools<70.0.0"

In [4]:
!pip install pytorch-lightning==2.3.0 omegaconf==2.3.0 \
            einops==0.7.0 kornia==0.7.3 natsort==8.4.0 \
            dctorch==0.1.2 lpips==0.1.4 scikit-image \
            rasterio packaging numpy==1.26.4 tqdm -q
print("Packages installed ✓")

Packages installed ✓


In [6]:
import subprocess, sys

# Install ONLY what's missing — never touch lightning/setuptools/numpy
safe_packages = [
    "einops==0.7.0",
    "kornia==0.7.3",
    "natsort==8.4.0",
    "dctorch==0.1.2",
    "lpips==0.1.4",
    "scikit-image",
    "rasterio",
    "packaging",
    "tqdm",
    "omegaconf==2.3.0",
]

for pkg in safe_packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg,
         "--no-deps", "-q"],
        capture_output=True, text=True
    )
    status = "✓" if result.returncode == 0 else "✗"
    print(f"{status} {pkg}")

# Verify critical imports
print("\nVerifying imports...")
for mod in ["torch", "rasterio", "omegaconf", "einops",
            "pytorch_lightning", "pkg_resources"]:
    try:
        __import__(mod)
        print(f"  ✓ {mod}")
    except ImportError as e:
        print(f"  ✗ {mod}: {e}")

✓ einops==0.7.0
✓ kornia==0.7.3
✓ natsort==8.4.0
✓ dctorch==0.1.2
✓ lpips==0.1.4
✓ scikit-image
✓ rasterio
✓ packaging
✓ tqdm
✓ omegaconf==2.3.0

Verifying imports...
  ✓ torch
  ✓ rasterio
  ✓ omegaconf
  ✓ einops
  ✓ pytorch_lightning
  ✓ pkg_resources


In [7]:
!pip install affine

In [8]:
# ── THE "ALL-IN-ONE" DEPENDENCY FIX ──────────────────────────────────────────
!pip install affine lazy_loader scipy networkx imageio tifffile cligj snuggs click

In [9]:
# Clone EMRDM
if not os.path.exists("/teamspace/studios/this_studio/EMRDM"):
    !git clone https://github.com/Ly403/EMRDM.git \
        /teamspace/studios/this_studio/EMRDM
    print("EMRDM cloned ✓")
else:
    print("EMRDM already exists ✓")

EMRDM already exists ✓


In [10]:
import os, sys, gc, time, json, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader
from datetime import datetime
from tqdm.notebook import tqdm
import rasterio
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn

# ── Paths ─────────────────────────────────────────────────────────────────────
EMRDM_PATH   = "/teamspace/studios/this_studio/EMRDM"
DATA_PATH    = "/teamspace/studios/this_studio/data/training_data"
WEIGHTS_PATH = "/teamspace/studios/this_studio/last.ckpt"
OUT_DIR      = "/teamspace/studios/this_studio/checkpoints"
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── sys.path + env ────────────────────────────────────────────────────────────
if EMRDM_PATH not in sys.path:
    sys.path.insert(0, EMRDM_PATH)
os.environ["USE_COMPILE"]  = "0"
os.environ["USE_FLASH_2"]  = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
warnings.filterwarnings("ignore",
    message="You are trying to `self.log()`")

print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# ── Apply all patches ─────────────────────────────────────────────────────────
def apply_patches():
    # flash_attn
    attn_path = f"{EMRDM_PATH}/sgm/modules/attention.py"
    with open(attn_path) as f: c = f.read()
    c = c.replace(
        "from flash_attn import flash_attn_qkvpacked_func, flash_attn_func",
        "try:\n    from flash_attn import flash_attn_qkvpacked_func, flash_attn_func\nexcept ImportError:\n    flash_attn_qkvpacked_func = None\n    flash_attn_func = None"
    ).replace(
        "from flash_attn.modules.mha import FlashSelfAttention",
        "try:\n    from flash_attn.modules.mha import FlashSelfAttention\nexcept ImportError:\n    FlashSelfAttention = None"
    )
    with open(attn_path, "w") as f: f.write(c)

    # natten
    for root, dirs, files in os.walk(EMRDM_PATH):
        for fname in files:
            if not fname.endswith(".py"): continue
            fp = os.path.join(root, fname)
            with open(fp) as f: c = f.read()
            orig = c
            lines = c.split("\n"); new_lines = []; i = 0
            while i < len(lines):
                line = lines[i]; strip = line.lstrip()
                indent = line[:len(line)-len(strip)]
                if strip == "import natten":
                    prev = new_lines[-1].strip() if new_lines else ""
                    if prev != "try:":
                        new_lines += [f"{indent}try:",
                                      f"{indent}    import natten",
                                      f"{indent}except ImportError:",
                                      f"{indent}    natten = None"]
                        i += 1; continue
                if strip.startswith("from natten"):
                    new_lines.append(f"{indent}# {strip}")
                    i += 1; continue
                new_lines.append(line); i += 1
            c = "\n".join(new_lines)
            if c != orig:
                with open(fp, "w") as f: f.write(c)

    # windowed attention
    img_trans = f"{EMRDM_PATH}/sgm/modules/diffusionmodules/k_diffusion/image_transformer.py"
    with open(img_trans) as f: c = f.read()
    old = ('        if natten is None:\n'
           '            raise ModuleNotFoundError("natten is required for neighborhood attention")')
    new = '''        if natten is None:
            n, h, w, C = x.shape
            q, k, v = qkv.chunk(3, dim=-1)
            ks    = min(self.kernel_size, h, w)
            pad_h = (ks - h % ks) % ks
            pad_w = (ks - w % ks) % ks
            if pad_h > 0 or pad_w > 0:
                q = torch.nn.functional.pad(q.permute(0,3,1,2),(0,pad_w,0,pad_h)).permute(0,2,3,1)
                k = torch.nn.functional.pad(k.permute(0,3,1,2),(0,pad_w,0,pad_h)).permute(0,2,3,1)
                v = torch.nn.functional.pad(v.permute(0,3,1,2),(0,pad_w,0,pad_h)).permute(0,2,3,1)
            ph, pw = h+pad_h, w+pad_w; nh, nw = ph//ks, pw//ks
            q = rearrange(q,"b (nh ks1)(nw ks2)(heads e)->(b nh nw)(ks1 ks2) heads e",nh=nh,nw=nw,ks1=ks,ks2=ks,e=self.d_head)
            k = rearrange(k,"b (nh ks1)(nw ks2)(heads e)->(b nh nw)(ks1 ks2) heads e",nh=nh,nw=nw,ks1=ks,ks2=ks,e=self.d_head)
            v = rearrange(v,"b (nh ks1)(nw ks2)(heads e)->(b nh nw)(ks1 ks2) heads e",nh=nh,nw=nw,ks1=ks,ks2=ks,e=self.d_head)
            q, k = scale_for_cosine_sim(q, k, self.scale[:, None], 1e-6)
            q2 = q.permute(0,2,1,3); k2 = k.permute(0,2,1,3); v2 = v.permute(0,2,1,3)
            out = torch.nn.functional.scaled_dot_product_attention(q2,k2,v2)
            out = out.permute(0,2,1,3)
            out = rearrange(out,"(b nh nw)(ks1 ks2) heads e->b (nh ks1)(nw ks2)(heads e)",b=n,nh=nh,nw=nw,ks1=ks,ks2=ks)
            if pad_h>0 or pad_w>0: out = out[:,:h,:w,:]
            x = self.out_proj(out)
            return x + skip'''
    if old in c:
        c = c.replace(old, new)
        with open(img_trans, "w") as f: f.write(c)
        print("Windowed attention patched ✓")
    else:
        print("Windowed attention already patched ✓")

apply_patches()
print("All patches applied ✓")
print("Imports done ✓")

Device : cuda
GPU    : NVIDIA H100 80GB HBM3
VRAM   : 85.0 GB
Windowed attention already patched ✓
All patches applied ✓
Imports done ✓


In [11]:

SCALE = 65535.0 / 906.0
NORM_STATS = {
    "optical": {
        "mean": [0.007363*SCALE, 0.007286*SCALE, 0.007609*SCALE],
        "std" : [0.001672*SCALE, 0.001723*SCALE, 0.00168 *SCALE],
    },
    "sar": {
        "mean": [-8.881173, -15.236059],
        "std" : [4.36772,    4.326354],
    },
}

def read_tif(path):
    with rasterio.open(path) as src:
        arr = src.read().astype(np.float32)
        
        if not np.isfinite(arr).all():
            arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
        return arr

def normalize(arr, mean, std):
    mean = np.array(mean, dtype=np.float32)[:, None, None]
    std  = np.array(std,  dtype=np.float32)[:, None, None]
    return (arr - mean) / (std + 1e-8)

def generate_cloud_mask(cloudy, clear, threshold=0.30):
    diff = np.abs(cloudy - clear).mean(axis=0, keepdims=True)
    mask = (diff > threshold).astype(np.float32)
    if mask.mean() < 0.05:
        mask = (diff > np.percentile(diff, 70)).astype(np.float32)
    return mask

def numpy_augment(arr):
    if np.random.random() > 0.5: arr = arr[:, ::-1, :].copy()
    if np.random.random() > 0.5: arr = arr[:, :, ::-1].copy()
    k = np.random.randint(0, 4)
    if k > 0: arr = np.rot90(arr, k=k, axes=(1,2)).copy()
    return arr


class LISSIVDataset(Dataset):
    
    def __init__(self, root, augment=True, max_dn=906.0):
        self.root    = root
        self.augment = augment
        self.max_dn  = max_dn
        self.stems   = sorted([
            f for f in os.listdir(os.path.join(root, "cloudy"))
            if f.endswith(".tif")
        ])
        print(f"Dataset: {len(self.stems)} samples | "
              f"mu=CLOUDY (fixed) | SAR+optical only")

    def __len__(self): return len(self.stems)
    def _p(self, folder, stem): return os.path.join(self.root, folder, stem)

    def __getitem__(self, idx):
        stem = self.stems[idx]

        # Raw reads
        cloudy_raw = np.clip(read_tif(self._p("cloudy",stem))/self.max_dn, 0, 1)
        clear_raw  = np.clip(read_tif(self._p("clear", stem))/self.max_dn, 0, 1)
        sar        = read_tif(self._p("sar", stem))

        # Cloud mask from raw difference
        mask = generate_cloud_mask(cloudy_raw, clear_raw)

        # Z-score normalization for conditioning
        cloudy_n = normalize(cloudy_raw, NORM_STATS["optical"]["mean"],
                                         NORM_STATS["optical"]["std"])
        sar_n    = normalize(sar,        NORM_STATS["sar"]["mean"],
                                         NORM_STATS["sar"]["std"])

        # Scale to [-1,1] for EMRDM target AND mu
        clear_11  = (clear_raw  * 2.0) - 1.0   # ground truth in [-1,1]
        cloudy_11 = (cloudy_raw * 2.0) - 1.0   # ← mu: cloudy in [-1,1]

        if self.augment:
            # Stack: cloudy_n(3) + clear_11(3) + cloudy_11(3) + sar_n(2) + mask(1) = 12
            all_ch = np.concatenate(
                [cloudy_n, clear_11, cloudy_11, sar_n, mask], axis=0)
            all_ch    = numpy_augment(all_ch)
            cloudy_n  = all_ch[0:3]
            clear_11  = all_ch[3:6]
            cloudy_11 = all_ch[6:9]
            sar_n     = all_ch[9:11]
            mask      = all_ch[11:12]

        # Conditioning: normalized cloudy(3) + SAR(2) = 5ch
        cond_t    = torch.from_numpy(
            np.concatenate([cloudy_n, sar_n], axis=0)).float()
        clear_t   = torch.from_numpy(clear_11).float()
        cloudy_t  = torch.from_numpy(cloudy_11).float()

        return {
            # EMRDM required keys
            "target"    : clear_t,        # [3,H,W] clean ground truth in [-1,1]
            "mu"        : cloudy_t,        # [3,H,W] CLOUDY in [-1,1] ← FIXED
            "S1"        : cond_t[3:5],    # [2,H,W] SAR
            "S2"        : cond_t[0:3],    # [3,H,W] normalized cloudy
            "S1S2"      : cond_t,         # [5,H,W] full conditioning
            # For metrics/visualization
            "clear_chw" : clear_t,
            "cloudy_11" : cloudy_t,        # cloudy in [-1,1] for inference mu
            "cloud_mask": torch.from_numpy(mask).float(),
        }


def build_dataloaders(base, batch_size=16, num_workers=4):
    full_ds = LISSIVDataset(base, augment=True)
    n       = len(full_ds)
    n_val   = int(n * 0.15); n_test = int(n * 0.15); n_train = n - n_val - n_test
    idx     = torch.randperm(n, generator=torch.Generator().manual_seed(42)).tolist()
    val_ds  = LISSIVDataset(base, augment=False)
    test_ds = LISSIVDataset(base, augment=False)

    train_loader = DataLoader(
        torch.utils.data.Subset(full_ds, idx[:n_train]),
        batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(
        torch.utils.data.Subset(val_ds, idx[n_train:n_train+n_val]),
        batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(
        torch.utils.data.Subset(test_ds, idx[n_train+n_val:]),
        batch_size=batch_size, shuffle=False,
        num_workers=num_workers, pin_memory=True)

    print(f"Train:{n_train} | Val:{n_val} | Test:{n_test}")
    return train_loader, val_loader, test_loader


# Sanity check
ds     = LISSIVDataset(DATA_PATH, augment=False)
sample = ds[0]
print(f"\nSanity check:")
print(f"  target   [3,H,W]: {sample['target'].shape}  "
      f"range [{sample['target'].min():.3f},{sample['target'].max():.3f}]  (clear)")
print(f"  mu       [3,H,W]: {sample['mu'].shape}  "
      f"range [{sample['mu'].min():.3f},{sample['mu'].max():.3f}]  (cloudy ← FIXED)")
print(f"  S1S2     [5,H,W]: {sample['S1S2'].shape}")
print(f"  mu != target    : {not torch.equal(sample['mu'], sample['target'])}  ← must be True")
print(f"  cloud_mask      : {sample['cloud_mask'].mean():.2%} coverage")
print("\nDataset ready ✓")

Dataset: 17278 samples | mu=CLOUDY (fixed) | SAR+optical only

Sanity check:
  target   [3,H,W]: torch.Size([3, 256, 256])  range [-0.874,-0.402]  (clear)
  mu       [3,H,W]: torch.Size([3, 256, 256])  range [-0.349,-0.161]  (cloudy ← FIXED)
  S1S2     [5,H,W]: torch.Size([5, 256, 256])
  mu != target    : True  ← must be True
  cloud_mask      : 29.64% coverage

Dataset ready ✓


In [ ]:
def sam_loss(pred, target, eps=1e-8):
    B, C, H, W = pred.shape
    p   = pred.reshape(B, C, -1)
    t   = target.reshape(B, C, -1)
    cos = (p*t).sum(1) / (p.norm(dim=1).clamp(eps) * t.norm(dim=1).clamp(eps))
    return torch.acos(cos.clamp(-1+eps, 1-eps)).mean()



def compute_metrics(pred, target, cloud_mask):
    B   = pred.shape[0]
    out = {"psnr":0.0,"ssim":0.0,"sam":0.0,"mae":0.0}
    cnt = {"psnr":0,  "ssim":0,  "sam":0,  "mae":0}

    p_np = np.clip((pred.detach().cpu().float().numpy()   + 1.0) / 2.0, 0, 1)
    t_np = np.clip((target.detach().cpu().float().numpy() + 1.0) / 2.0, 0, 1)
    m_np = cloud_mask.detach().cpu().numpy()

    for b in range(B):
        p = p_np[b].transpose(1,2,0)
        t = t_np[b].transpose(1,2,0)
        m = m_np[b,0]

        if not (np.isfinite(p).all() and np.isfinite(t).all()):
            continue

        out["psnr"] += psnr_fn(t, p, data_range=1.0); cnt["psnr"] += 1
        out["ssim"] += ssim_fn(t, p, data_range=1.0, channel_axis=2, win_size=7); cnt["ssim"] += 1
        if m.sum() > 0:
            pc = p[m>0]; tc = t[m>0]
            out["mae"] += float(np.abs(pc-tc).mean()); cnt["mae"] += 1
            dot = (pc*tc).sum(1)
            pn  = np.linalg.norm(pc,axis=1).clip(1e-8)
            tn  = np.linalg.norm(tc,axis=1).clip(1e-8)
            cos = np.clip(dot/(pn*tn), -1+1e-8, 1-1e-8)
            out["sam"] += float(np.degrees(np.arccos(cos).mean())); cnt["sam"] += 1

    return {k: (out[k] / cnt[k] if cnt[k] > 0 else 0.0) for k in out}


print("Loss + metrics ready (compute_metrics denormalization bug FIXED) ✓")

In [13]:
def load_checkpoint(path, model, optimizer=None, scheduler=None):
    print(f"Loading: {path}")
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    if optimizer and "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    if scheduler and "scheduler_state" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state"])
    epoch   = ckpt.get("epoch", 0)
    history = ckpt.get("history",
              {"train_loss":[],"val_psnr":[],"val_ssim":[],"val_sam":[],"val_mae":[]})
    print(f"Resumed from epoch {epoch}")
    if ckpt.get("metrics"):
        m = ckpt["metrics"]
        print(f"Last metrics: PSNR={m.get('psnr',0):.4f} "
              f"SSIM={m.get('ssim',0):.4f} SAM={m.get('sam',0):.4f}°")
    return epoch+1, history


def save_checkpoint(path, epoch, model, optimizer, scheduler,
                    history, hyperparams, metrics, is_best=False):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({
        "epoch"          : epoch,
        "model_state"    : model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "history"        : history,
        "metrics"        : metrics,
        "hyperparams"    : hyperparams,
        "saved_at"       : datetime.now().isoformat(),
    }, path)
    json_path = path.replace(".pth","_info.json")
    with open(json_path,"w") as f:
        json.dump({
            "epoch"      : epoch,
            "metrics"    : metrics,
            "hyperparams": hyperparams,
            "saved_at"   : datetime.now().isoformat(),
        }, f, indent=2)
    tag = " BEST" if is_best else ""
    print(f"  Saved{tag}: {os.path.basename(path)}")


def list_checkpoints(out_dir=OUT_DIR):
    if not os.path.exists(out_dir):
        print("No checkpoints yet"); return
    files = sorted([f for f in os.listdir(out_dir) if f.endswith("_info.json")])
    if not files: print("No checkpoints yet"); return
    print(f"\n{'File':<28} {'Ep':>4} {'PSNR':>8} {'SSIM':>7} {'SAM':>7}")
    print("─"*58)
    for jf in files:
        with open(os.path.join(out_dir,jf)) as f: info = json.load(f)
        m = info.get("metrics",{})
        print(f"{jf.replace('_info.json',''):28} "
              f"{info.get('epoch',0):>4} "
              f"{m.get('psnr',0):>8.3f} "
              f"{m.get('ssim',0):>7.4f} "
              f"{m.get('sam',0):>7.3f}")

print("Checkpoint utilities ready ✓")

Checkpoint utilities ready ✓


In [14]:
!pip install opencv-python-headless

  Using cached numpy-2.5.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached numpy-2.5.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
matplotlib 3.8.2 requires numpy<2,>=1.21, but you have numpy 2.5.0 which is incompatible.
scipy 1.11.4 requires numpy<1.28.0,>=1.21.6, but you have numpy 2.5.0 which is incompatible.
dctorch 0.1.2 requires numpy<2.0.0,>=1.22.3, but you have numpy 2.5.0 which is incompatible.
pandas 2.1.4 requires numpy<2,>=1.26.0; python_version >= "3.12", but you have numpy 2.5.0 which is incompatible.
scikit-learn 1.3.2 requires numpy<2.0,>=1.17.3, but you have numpy 2.5.0 which is in

In [ ]:
def build_model():
    import importlib
    for key in list(sys.modules.keys()):
        if "sgm" in key: del sys.modules[key]
    importlib.invalidate_caches()

    from sgm.util import instantiate_from_config
    from omegaconf import OmegaConf
    from sgm.modules.diffusionmodules.k_diffusion import flags

    
    flags.state.checkpointing = True
    print("Gradient checkpointing: ON")

    cfg = OmegaConf.load(f"{EMRDM_PATH}/configs/example_training/sentinel.yaml")
    cfg.model.params.network_config.params.in_channels  = 8
    cfg.model.params.network_config.params.out_channels = 3
    model = instantiate_from_config(cfg.model).to(DEVICE)
    print("EMRDM built ✓")

    if os.path.exists(WEIGHTS_PATH):
        print(f"Loading: {os.path.basename(WEIGHTS_PATH)}")
        ckpt = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=False)
        sd       = ckpt.get("state_dict", ckpt)
        model_sd = model.state_dict()
        resized  = 0
        for key in list(sd.keys()):
            if key not in model_sd: continue
            ts = model_sd[key].shape
            if sd[key].shape == ts: continue
            if sd[key].ndim != len(ts): del sd[key]; continue
            new_t  = torch.zeros(ts, dtype=sd[key].dtype)
            slices = tuple(slice(0,min(o,n)) for o,n in zip(sd[key].shape,ts))
            new_t[slices] = sd[key][slices]; sd[key] = new_t; resized += 1
        missing, unexpected = model.load_state_dict(sd, strict=False)
        print(f"Weights: Resized={resized} Missing={len(missing)} "
              f"Unexpected={len(unexpected)}")
    else:
        print("No pretrained weights — training from scratch")

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameters: {n_params/1e6:.2f}M")

    
    test_b = {k: torch.randn(2, ch, 256, 256).to(DEVICE)
              for k, ch in [("target",3),("mu",3),("S1",2),("S2",3),("S1S2",5)]}
    try:
        model.train()
        with torch.no_grad():
            loss = model.training_step(test_b, 0)
        if isinstance(loss, dict): loss = loss["loss"]
        print(f"training_step test: loss={loss.item():.4f} ✓")
        print("Model ready ✓")
    except Exception as e:
        print(f"Test failed: {e}")
        import traceback; traceback.print_exc()

    return model

gc.collect(); torch.cuda.empty_cache()
model = build_model()

Gradient checkpointing: ON


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Initialized embedder #0: IndentityEmbedder with 0 params. Trainable: True
Keeping EMAs of 121.
EMRDM built ✓
Loading: last.ckpt
Weights: Resized=4 Missing=0 Unexpected=0
Parameters: 39.13M
training_step test: loss=7.5739 ✓
Model ready ✓


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torch/utils/checkpoint.py:85: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pytorch_lightning/core/module.py:447: You are trying to `self.log()` but the `self.trainer` reference is not registered on the model yet. This is most likely because the model hasn't been passed to the `Trainer`


In [ ]:
import contextlib, torch, os, numpy as np
import matplotlib.pyplot as plt

@contextlib.contextmanager
def _noop_ema_scope(self, *args, **kwargs):
    yield


METHOD_USAGE = {"method_1": 0, "method_2": 0, "method_3": 0, "fallback_cloudy": 0}
_seen_errors = set()

def _log_once(tag, e):
    key = (tag, str(e))
    if key not in _seen_errors:
        _seen_errors.add(key)
        print(f"  [get_prediction] {tag} failed: {e}")

def get_prediction(model, batch, n_steps=30):
    model.eval()
    cond      = batch["S1S2"].to(DEVICE)
    cloudy_11 = batch["cloudy_11"].to(DEVICE)
    B         = cond.shape[0]

    log_batch = {
        "target": batch["target"][:B].to(DEVICE),
        "mu"    : cloudy_11,
        "S1"    : cond[:, 3:5],
        "S2"    : cond[:, 0:3],
        "S1S2"  : cond,
    }

    with torch.no_grad():
        # ── Method 1: Patch out EMA so log_images uses fine-tuned weights ────
        try:
            import types
            original_ema_scope = model.ema_scope
            model.ema_scope = types.MethodType(_noop_ema_scope, model)
            try:
                samples = model.log_images(
                    log_batch, N=B, sample=True,
                    ddim_steps=n_steps, return_keys=["samples"],
                )
                pred = samples.get("samples", None)
            finally:
                model.ema_scope = original_ema_scope

            if pred is not None:
                if pred.ndim == 4 and pred.shape[-1] == 3:
                    pred = pred.permute(0, 3, 1, 2)
                pred = pred.to(DEVICE).clamp(-1, 1)
                bad  = ~torch.isfinite(pred).flatten(1).all(1)
                if bad.any():
                    pred[bad] = cloudy_11[bad]
                METHOD_USAGE["method_1"] += 1
                return pred
        except Exception as e:
            _log_once("Method 1", e)

        try:
            cond_dict = model.conditioner(log_batch)
            mu_enc = model.encode_first_stage(cloudy_11)
            x = mu_enc + torch.randn_like(mu_enc) * 0.3

            x_denoised = model.sampler(
                model.denoiser,
                x,
                cond_dict,
                model  = model.model,
                x_T    = None,
                **{"sigma2st": model.sigma2st}
                    if hasattr(model, "sigma2st") else {}
            )
            pred = model.decode_first_stage(x_denoised).clamp(-1, 1)
            if torch.isfinite(pred).all():
                METHOD_USAGE["method_2"] += 1
                return pred
        except Exception as e:
            _log_once("Method 2", e)

        # ── Method 3: Step-by-step denoising via denoiser directly ───────────
        try:
            cond_dict = model.conditioner(log_batch)
            mu_enc    = model.encode_first_stage(cloudy_11)
            x         = mu_enc + torch.randn_like(mu_enc) * 0.3

            sigmas = torch.cos(
                torch.linspace(0, torch.pi/2, n_steps)
            ).to(DEVICE) * 0.3 + 0.01

            for sig in sigmas:
                sig_b = sig.expand(B)
                st    = model.sigma2st(sig_b)
                x     = model.denoiser(model.model, x, sig_b, cond_dict, st)
                x     = x.clamp(-3, 3)

            pred = model.decode_first_stage(x).clamp(-1, 1)
            if torch.isfinite(pred).all():
                METHOD_USAGE["method_3"] += 1
                return pred
        except Exception as e:
            _log_once("Method 3", e)

    # Last resort: return cloudy (finite, visible)
    METHOD_USAGE["fallback_cloudy"] += 1
    return cloudy_11


def compute_val_reconstruction_loss(model, val_loader, max_batches=30):
    model.eval()
    total = 0.0; nb = 0
    with torch.no_grad():
        for i, vb in enumerate(val_loader):
            if i >= max_batches: break
            pred = get_prediction(model, vb, n_steps=10)
            tgt  = vb["clear_chw"].to(DEVICE)
            if torch.isfinite(pred).all():
                total += torch.nn.functional.l1_loss(pred, tgt).item()
                nb += 1
    return total / max(nb, 1)


def denorm_optical(t):
    arr = t.cpu().float().numpy()
    if arr.ndim == 4: arr = arr.transpose(0,2,3,1)
    elif arr.ndim == 3: arr = arr.transpose(1,2,0)
    return np.clip((arr + 1.0) / 2.0, 0, 1)


def show_epoch_results(model, val_loader, epoch, history, out_dir=OUT_DIR):
    os.makedirs(out_dir, exist_ok=True)
    if history["val_psnr"]:
        print(f"\n  {'Metric':<8} {'Epoch':>8} {'Best':>8} {'Target':>8}  Status")
        print(f"  {'─'*52}")
        rows = [
            ("PSNR",  history["val_psnr"],        "↑", 28.0),
            ("SSIM",  history["val_ssim"],        "↑",  0.88),
            ("SAM°",  history["val_sam"],         "↓",  2.0),
            ("MAE",   history["val_mae"],         "↓",  0.025),
            ("VRecon",history.get("val_recon_loss",[]),"↓", None),
        ]
        for name, vals, direction, target in rows:
            if not vals: continue
            cur  = vals[-1]
            best = max(vals) if direction=="↑" else min(vals)
            if target:
                gap    = cur - target if direction=="↑" else target - cur
                status = "✓ TARGET MET" if gap >= 0 else f"{abs(gap):.4f} to go"
            else:
                status = ""
            print(f"  {name:<8} {cur:>8.4f} {best:>8.4f} "
                  f"{str(target) if target else '':>8}  {status}")

    # NEW: surface which inference path is actually being used, so a
    # silent fallback to the crude Methods 2/3 samplers is visible instead
    # of hiding behind seemingly-fine metrics.
    print(f"  Inference method usage so far: {METHOD_USAGE}")

    model.eval()
    batch = next(iter(val_loader))
    n     = 3
    pred  = get_prediction(model, batch, n_steps=30)[:n]
    inp   = batch["cloudy_11"][:n]
    tgt   = batch["clear_chw"][:n]
    msk   = batch["cloud_mask"][:n]

    cloudy_rgb = denorm_optical(inp)
    pred_rgb   = denorm_optical(pred.cpu())
    gt_rgb     = denorm_optical(tgt)
    mask_np    = msk[:,0].numpy()

    fig, axes = plt.subplots(n, 4, figsize=(16, 4*n))
    fig.suptitle(f"Epoch {epoch} — EMRDM LISS-IV (non-EMA inference)",
                 fontsize=12, fontweight="bold")
    titles = ["Cloudy Input","Cloud Mask","EMRDM Predicted","Ground Truth"]
    for i in range(n):
        m = compute_metrics(pred[i:i+1].to(DEVICE),
                            tgt[i:i+1].to(DEVICE),
                            msk[i:i+1].to(DEVICE))
        for j,(img,cmap) in enumerate(zip(
            [cloudy_rgb[i],mask_np[i],pred_rgb[i],gt_rgb[i]],
            [None,"gray",None,None]
        )):
            axes[i][j].imshow(img, cmap=cmap, vmin=0, vmax=1)
            axes[i][j].axis("off")
            if i==0: axes[i][j].set_title(titles[j], fontweight="bold", fontsize=10)
        axes[i][2].set_xlabel(
            f"PSNR={m['psnr']:.2f}  SSIM={m['ssim']:.3f}  "
            f"SAM={m['sam']:.2f}°  MAE={m['mae']:.4f}",
            fontsize=8, color="darkgreen")
    plt.tight_layout()
    path = os.path.join(out_dir, f"epoch_{epoch:03d}_pred.png")
    plt.savefig(path, dpi=120, bbox_inches="tight")
    plt.show(); plt.close()
    print(f"  Saved: epoch_{epoch:03d}_pred.png")


def plot_history(history, out_dir=OUT_DIR):
    if not history["train_loss"]: return
    keys    = ["train_loss","val_recon_loss","val_psnr","val_ssim","val_sam","val_mae"]
    targets = [None, None, 28.0, 0.88, 2.0, 0.025]
    colors  = ["steelblue","navy","green","orange","red","purple"]
    arrows  = ["↓","↓","↑","↑","↓","↓"]
    titles  = ["Train Loss","Val Recon L1","Val PSNR dB",
               "Val SSIM","Val SAM°","Val MAE"]
    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    axes = axes.flatten()
    for ax, key, color, arrow, title, target in zip(
            axes, keys, colors, arrows, titles, targets):
        vals = history.get(key, [])
        if not vals: ax.set_visible(False); continue
        ep = list(range(1, len(vals)+1))
        ax.plot(ep, vals, color=color, linewidth=2, marker="o", markersize=5)
        ax.set_title(f"{title} {arrow}", fontweight="bold")
        ax.set_xlabel("Epoch"); ax.grid(True, alpha=0.3)
        best = min(vals) if "↓" in arrow else max(vals)
        ax.axhline(best, color=color, linestyle="--", alpha=0.5,
                   label=f"Best:{best:.4f}")
        if target:
            ax.axhline(target, color="black", linestyle=":", alpha=0.8,
                       label=f"Target:{target}")
        ax.legend(fontsize=7)
    plt.suptitle("EMRDM Fine-tuning — LISS-IV (non-EMA inference fix)",
                 fontweight="bold", fontsize=13)
    plt.tight_layout()
    path = os.path.join(out_dir, "training_curves.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show(); plt.close()
    print(f"Curves → {path}")

print("Inference fixed ✓ (EMA bypassed, fallback methods now instrumented + visible)")

In [ ]:


ckpt_path = os.path.join(OUT_DIR, "best_model.pth")
assert os.path.exists(ckpt_path), f"Checkpoint not found: {ckpt_path}"

ckpt = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"Loaded checkpoint from epoch {ckpt.get('epoch')}")
print(f"Checkpoint's OWN recorded metrics (buggy): {ckpt.get('metrics')}")

_, val_loader, _ = build_dataloaders(DATA_PATH, batch_size=16, num_workers=4)

METHOD_USAGE["method_1"] = METHOD_USAGE["method_2"] = 0
METHOD_USAGE["method_3"] = METHOD_USAGE["fallback_cloudy"] = 0

vm = {"psnr":0.0,"ssim":0.0,"sam":0.0,"mae":0.0}; nb_ = 0
with torch.no_grad():
    for vb in tqdm(val_loader, desc="Re-validating with FIXED metrics"):
        pred = get_prediction(model, vb, n_steps=20)
        tgt  = vb["clear_chw"].to(DEVICE)
        msk  = vb["cloud_mask"].to(DEVICE)
        if not torch.isfinite(pred).all():
            continue
        m = compute_metrics(pred, tgt, msk)
        for k in vm:
            if not np.isnan(m[k]): vm[k] += m[k]
        nb_ += 1
for k in vm: vm[k] /= max(nb_, 1)

print("\n" + "="*58)
print("REAL (corrected) validation metrics for best_model.pth")
print("="*58)
print(f"  PSNR : {vm['psnr']:.4f} dB   (target ≥ 28)")
print(f"  SSIM : {vm['ssim']:.4f}       (target ≥ 0.88)")
print(f"  SAM  : {vm['sam']:.4f}°       (target ≤ 2.0)")
print(f"  MAE  : {vm['mae']:.6f}     (target ≤ 0.025)")
print("="*58)
print(f"Inference method usage during this validation pass: {METHOD_USAGE}")
print("If method_2/method_3/fallback_cloudy are non-zero, Method 1")
print("(the real model.log_images sampler) is failing silently somewhere")
print("-- check the '[get_prediction] Method 1 failed: ...' lines above,")
print("which are now printed instead of being swallowed.")

In [ ]:

LIVE_HISTORY = {}

def train_emrdm(
    epochs      = 30,
    batch_size  = 16,
    lr          = 5e-5,
    num_workers = 4,
    save_every  = 2,
    resume_from = None,
):
    global LIVE_HISTORY
    os.makedirs(OUT_DIR, exist_ok=True)
    torch.cuda.empty_cache()

    hyperparams = {
        "epochs":epochs, "batch_size":batch_size, "lr":lr,
        "mu":"cloudy image in [-1,1]",
        "inference":"non-EMA DDIM + direct denoiser fallback",
    }

    train_loader, val_loader, test_loader = build_dataloaders(
        DATA_PATH, batch_size=batch_size, num_workers=num_workers)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr*0.01)

    history = {
        "train_loss":[], "val_recon_loss":[],
        "val_psnr":[], "val_ssim":[], "val_sam":[], "val_mae":[]
    }
    start_epoch = 1
    best_psnr   = 0.0

    if resume_from and os.path.exists(resume_from):
        start_epoch, history = load_checkpoint(
            resume_from, model, optimizer, scheduler)
        if "val_recon_loss" not in history:
            history["val_recon_loss"] = [0.0]*len(history.get("train_loss",[]))
        best_psnr = max(history["val_psnr"]) if history["val_psnr"] else 0.0
        for _ in range(start_epoch-1): scheduler.step()

    # FIX: bind the live global to whichever `history` dict we ended up
    # with (fresh or resumed) -- BEFORE the training loop starts, so it's
    # available immediately, not just after the first epoch completes.
    LIVE_HISTORY = history

    n_batches     = len(train_loader)
    session_start = time.time()

    print(f"\nEMRDM Fine-tuning — LISS-IV")
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"Epochs   : {start_epoch} → {epochs} | Batch: {batch_size}")
    print(f"mu       : cloudy image (fixed)")
    print(f"Targets  : PSNR≥28 | SSIM≥0.88 | SAM≤2.0 | MAE≤0.025\n")

    import warnings
    warnings.filterwarnings("ignore", message="You are trying to `self.log()`")
    warnings.filterwarnings("ignore", message="None of the inputs have requires_grad")

    for epoch in range(start_epoch, epochs+1):
        t0 = time.time()

        # ── Train ─────────────────────────────────────────────────────────────
        model.train()
        train_loss = 0.0; valid_steps = 0

        pbar = tqdm(train_loader,
                    desc=f"Epoch {epoch}/{epochs}",
                    total=n_batches, leave=True)

        for batch in pbar:
            sgm_batch = {k: batch[k].to(DEVICE)
                         for k in ["target","mu","S1","S2","S1S2"]}

            if torch.equal(sgm_batch["mu"], sgm_batch["target"]):
                continue
            if any(torch.isnan(v).any() or torch.isinf(v).any()
                   for v in sgm_batch.values()):
                continue

            optimizer.zero_grad(set_to_none=True)
            loss = model.training_step(sgm_batch, 0)
            if isinstance(loss, dict): loss = loss["loss"]
            if torch.isnan(loss) or torch.isinf(loss): continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            optimizer.step()
            train_loss += loss.item(); valid_steps += 1

            if valid_steps % 100 == 0:
                free, tv = torch.cuda.mem_get_info()
                pbar.set_postfix({
                    "loss": f"{train_loss/valid_steps:.4f}",
                    "vram": f"{(tv-free)/1e9:.0f}GB"
                })

        scheduler.step()
        avg_loss = train_loss / max(valid_steps, 1)

        # ── Validate ──────────────────────────────────────────────────────────
        model.eval()
        vm = {"psnr":0.0,"ssim":0.0,"sam":0.0,"mae":0.0}; nb=0

        with torch.no_grad():
            for vb in tqdm(val_loader, desc=f"  Val {epoch}", leave=False):
                pred = get_prediction(model, vb, n_steps=20)
                tgt  = vb["clear_chw"].to(DEVICE)
                msk  = vb["cloud_mask"].to(DEVICE)
                if not torch.isfinite(pred).all(): continue
                m = compute_metrics(pred, tgt, msk)
                for k in vm:
                    if not np.isnan(m[k]): vm[k] += m[k]
                nb += 1

        for k in vm: vm[k] /= max(nb, 1)

        # Real reconstruction val loss (L1 on a few batches — fast)
        val_recon = compute_val_reconstruction_loss(model, val_loader, max_batches=30)

        elapsed = (time.time()-t0)/60
        total   = (time.time()-session_start)/60
        free, tv = torch.cuda.mem_get_info()
        star    = ""

        
        history["train_loss"].append(avg_loss)
        history["val_recon_loss"].append(val_recon)
        history["val_psnr"].append(vm["psnr"])
        history["val_ssim"].append(vm["ssim"])
        history["val_sam"].append(vm["sam"])
        history["val_mae"].append(vm["mae"])
        LIVE_HISTORY = history  # keep the global mirror current every epoch

        if vm["psnr"] > best_psnr:
            best_psnr = vm["psnr"]; star = " ★ NEW BEST"
            save_checkpoint(
                os.path.join(OUT_DIR,"best_model.pth"),
                epoch, model, optimizer, scheduler,
                history, hyperparams, vm, is_best=True)

        if epoch % save_every == 0:
            save_checkpoint(
                os.path.join(OUT_DIR, f"epoch_{epoch:03d}.pth"),
                epoch, model, optimizer, scheduler,
                history, hyperparams, vm)

        # ── Per-epoch display ─────────────────────────────────────────────────
        print(f"\n{'═'*58}")
        print(f"  EPOCH {epoch}/{epochs}{star}")
        print(f"{'─'*58}")
        print(f"  Train Loss (diffusion) : {avg_loss:.4f}")
        print(f"  Val Recon Loss (L1)    : {val_recon:.4f}  ← real reconstruction quality")
        print(f"{'─'*58}")

        metrics_info = [
            ("PSNR",   vm["psnr"], "↑", 28.0),
            ("SSIM",   vm["ssim"], "↑",  0.88),
            ("SAM°",   vm["sam"],  "↓",  2.0),
            ("MAE",    vm["mae"],  "↓",  0.025),
        ]
        for name, val, direction, target in metrics_info:
            gap    = val - target if direction=="↑" else target - val
            status = "✓ TARGET MET" if gap >= 0 else f"  {abs(gap):.4f} to go"
            print(f"  {name:<6}: {val:>8.4f}   target {direction}{target}   {status}")

        print(f"{'─'*58}")
        print(f"  Time : {elapsed:.1f} min | Total: {total:.1f} min")
        print(f"  VRAM : {(tv-free)/1e9:.1f}/{tv/1e9:.0f} GB")
        print(f"{'─'*58}")

        # Per-epoch visualization
        show_epoch_results(model, val_loader, epoch, history)

        if epoch % 2 == 0 or epoch == epochs:
            plot_history(history)

        print(f"{'═'*58}\n")

    # Final test
    print("="*58)
    print("FINAL TEST RESULTS — best checkpoint")
    print("="*58)
    ckpt = torch.load(os.path.join(OUT_DIR,"best_model.pth"), weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    tm = {"psnr":0.0,"ssim":0.0,"sam":0.0,"mae":0.0}; nt=0
    with torch.no_grad():
        for tb in tqdm(test_loader, desc="Testing"):
            pred = get_prediction(model, tb, n_steps=50)
            tgt  = tb["clear_chw"].to(DEVICE)
            msk  = tb["cloud_mask"].to(DEVICE)
            if not torch.isfinite(pred).all(): continue
            m = compute_metrics(pred, tgt, msk)
            for k in tm:
                if not np.isnan(m[k]): tm[k] += m[k]
            nt += 1
    for k in tm: tm[k] /= max(nt, 1)
    print(f"  PSNR : {tm['psnr']:.4f} dB   (target ≥28)")
    print(f"  SSIM : {tm['ssim']:.4f}       (target ≥0.88)")
    print(f"  SAM  : {tm['sam']:.4f}°       (target ≤2.0)")
    print(f"  MAE  : {tm['mae']:.6f}     (target ≤0.025)")
    print("="*58)
    return model, history

print("Training function ready ✓ (history append/save order fixed, LIVE_HISTORY exposed)")

In [ ]:
import torch, gc
gc.collect(); torch.cuda.empty_cache()


try:
    model, history = train_emrdm(
        epochs      = 30,
        batch_size  = 16,
        lr          = 5e-5,
        num_workers = 4,
        save_every  = 2,
        resume_from = f"{OUT_DIR}/best_model.pth",
    )
except KeyboardInterrupt:
    print("\nTraining interrupted — recovering in-progress history from LIVE_HISTORY.")
    history = LIVE_HISTORY

In [ ]:

import os, glob, torch

def _get_history():
    if "history" in globals() and history:
        return history, "in-memory `history` (train_emrdm returned normally)"
    if "LIVE_HISTORY" in globals() and LIVE_HISTORY and LIVE_HISTORY.get("train_loss"):
        return LIVE_HISTORY, "LIVE_HISTORY (recovered mid-run / after interrupt)"

    ckpt_path = os.path.join(OUT_DIR, "best_model.pth")
    if not os.path.exists(ckpt_path):
        all_ckpts = glob.glob(os.path.join(OUT_DIR, "epoch_*.pth"))
        if not all_ckpts:
            raise RuntimeError("No history in memory and no checkpoints on disk.")
        ckpt_path = max(all_ckpts, key=os.path.getmtime)

    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    return ckpt["history"], f"disk checkpoint: {os.path.basename(ckpt_path)} (epoch {ckpt.get('epoch')})"

history, source = _get_history()
print(f"history source: {source}\n")

for k in ["train_loss","val_recon_loss","val_psnr","val_ssim","val_sam","val_mae"]:
    vals = history.get(k, [])
    print(k, [round(v, 6) for v in vals])

In [ ]:
import os, torch

def check_checkpoint_health(path):
    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        sd   = ckpt.get("model_state", ckpt)
        n_bad = sum(
            1 for v in sd.values()
            if torch.is_tensor(v) and v.is_floating_point() and not torch.isfinite(v).all()
        )
        ok = n_bad == 0
        ep = ckpt.get("epoch", "?")
        m  = ckpt.get("metrics", {})
        print(f"{'✓ OK ' if ok else '✗ NaN'}  epoch {ep!s:>3}  "
              f"PSNR={m.get('psnr',0):>7.3f}  SSIM={m.get('ssim',0):.3f}  "
              f"bad_tensors={n_bad:<4}  {os.path.basename(path)}")
        return ok
    except Exception as e:
        print(f"✗ ERR  {os.path.basename(path)}: {e}")
        return False

print(f"{'Status':<8}\n" + "─"*70)
ckpt_files = sorted([f for f in os.listdir(OUT_DIR) if f.endswith(".pth")])
last_good = None
for f in ckpt_files:
    path = os.path.join(OUT_DIR, f)
    if check_checkpoint_health(path):
        last_good = path

print("\n" + "─"*70)
if last_good:
    print(f"Last known-good checkpoint: {last_good}")
    print("→ Use this as resume_from once the training loop below is patched.")
else:
    print("No clean checkpoint found — every saved checkpoint already has NaN weights.")
    print("→ You'll need to rebuild the model from the pretrained .ckpt and start over,")
    print("  but WITH the gradient guard below in place this time.")